# preprocessing.ipynb
# Run on Colab. Loads raw data from GitHub, cleans it, saves 3 slices.

import

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

load raw data from github

In [2]:
URL = "https://raw.githubusercontent.com/keavenq/ecs171-group2-rain-australia/main/data/weatherAUS.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

(145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


drop rows where target is missing

In [3]:
df = df.dropna(subset=["RainTomorrow"])
print(df.shape)

(142193, 23)


add Month column for season filtering

In [4]:
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month

drop rows with any remaining missing values

In [5]:
df = df.dropna()
print(df.shape)

(56420, 24)


encode target and RainToday as 0/1

In [6]:
df["RainTomorrow"] = (df["RainTomorrow"] == "Yes").astype(int)
df["RainToday"] = (df["RainToday"] == "Yes").astype(int)

one-hot encode the categorical columns

In [7]:
df = pd.get_dummies(df, columns=["Location", "WindGustDir", "WindDir9am", "WindDir3pm"])


drop Date (already used to make Month)

In [8]:
df = df.drop(columns=["Date"])


standardize numeric features
(don't scale the target, the binary RainToday, the Month, or the one-hot columns)

In [9]:
num_cols = ["MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
            "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
            "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
            "Cloud9am", "Cloud3pm", "Temp9am", "Temp3pm"]
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

make the 3 slices

In [10]:
summer = df[df["Month"].isin([12, 1, 2])].drop(columns=["Month"])
winter = df[df["Month"].isin([6, 7, 8])].drop(columns=["Month"])
combined = df[df["Month"].isin([12, 1, 2, 6, 7, 8])].drop(columns=["Month"])

print("summer:", summer.shape)
print("winter:", winter.shape)
print("combined:", combined.shape)

summer: (13760, 92)
winter: (14002, 92)
combined: (27762, 92)


save the 3 slices

In [11]:
summer.to_csv("summer.csv", index=False)
winter.to_csv("winter.csv", index=False)
combined.to_csv("combined.csv", index=False)